<a href="https://colab.research.google.com/github/paulagirones/Aerogels_thermal_conductivity/blob/main/aerogels_thermal_conductivity_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerogels thermal conductivity

# DATA EXPLORATION

In [25]:
import pandas as pd
import numpy as np

# Load the dataset
df_raw = pd.read_csv('20250220_dataset.csv')

# Working copy
df = df_raw.copy()
# Inspect the first rows
print("First 5 rows:")
display(df.head())

# Display information about the dataset
print("\nDataset information:")
df.info()



First 5 rows:


,Entry,Drying,Purpose,Shape,First materials,Second materials,Modification,First material class,Second materials class,Modification class,...,BET surface area m2/g,BJH pore size nm,BJH pore volume (total) cm3/g,compression elastic modulus MPa,SEM,Thermal conductivity W/(m K),Thermal method,Thermal method detail,Unnamed: 28,Unnamed: 29
0,1,SCD,Not specific,Monolith?,Soda soap,NaN,NaN,Q,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16559 entries, 0 to 16558
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Entry                            16559 non-null  int64  
 1   Drying                           16559 non-null  object 
 2   Purpose                          16559 non-null  object 
 3   Shape                            16559 non-null  object 
 4   First materials                  16559 non-null  object 
 5   Second materials                 6368 non-null   object 
 6   Modification                     3086 non-null   object 
 7   First material class             16559 non-null  object 
 8   Second materials class           6368 non-null   object 
 9   Modification class               3082 non-null   object 
 10  Gelation class                   16559 non-null  object 
 11  Solvent class                    16559 non-null  object 
 

# CLEANING AND PRE-PROCESSING

In [26]:
# Check Porosity type
print('Porosity type:')
print(df['Porosity'].dtype)
print(df['Porosity'].unique()[:20])

# Check SEM type
print('SEM type:')
print(df['SEM'].dtype)
print(df['SEM'].unique()[:20])


# Check Unnamed:29 type and content
print('Unnamed: 29 column content:')
print(df['Unnamed: 29'].dtype)
print(df['Unnamed: 29'].unique())


Porosity type:
object
[nan '95%' '94%' '98%' '96%' '97%' '65%' '74.38%' '80.75%' '84.19%'
 '88.06%' '87.45%' '82.64%' '78.32%' '67.97%' '32.80%' '48.20%' '53.70%'
 '59.30%' '65.90%']
SEM type:
object
[nan 'Mesoporous (nanoparticulate)' 'Macroporous (cellular)' 'Mesoporous?'
 'Macroporous (flakes)' 'Macroporous (particulate)'
 'Macroporous (fibrous)' 'Macroporous (fibrous and particulate)'
 'Mesoporous (nanofibrous)' 'Microparticles'
 'Macroporous (cellular and fibrous)'
 'Mesoporous (nanofibrous and nanoparticulate)'
 'Macroporous (cellular and flakes)' 'Macroporous (fibrous and flakes)'
 'Macroporous (nanoparticulate)' 'Macroporous (cellular and particulate)'
 'Macroporous (flakes and particulate)' 'Macroporous' 'Macroporous?'
 'Macroporous (nanofibrous)']
Unnamed: 29 column content:
object
[nan 'kokoko' 'koko' 'kokokoko' 'kokokokoko']


In [27]:
#Fix porosity type (it is numeric)
df['Porosity'] = (
    df['Porosity']
    .str.replace('%', '', regex=False)
    .astype(float)
)

display(df['Porosity'].describe())

# Remove empty and wrong columns
df = df.drop(columns=['Unnamed: 28', 'Unnamed: 29'])

,Porosity
count,7122.000000
mean,87.895411
std,14.065184
min,0.530000
25%,85.000000
50%,92.500000
75%,97.100000
max,100.000000


In [28]:
# Check missing values
missing = pd.DataFrame({
    'Missing values': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100
})
display(missing.sort_values('Missing (%)', ascending=False))

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

,Missing values,Missing (%)
Post-drying cross-linking,16064,97.010689
BJH pore size nm,15547,93.888520
BJH pore volume (total) cm3/g,14080,85.029289
Thermal conductivity W/(m K),14075,84.999094
Thermal method detail,14069,84.962860
Thermal method,14066,84.944743
compression elastic modulus MPa,13558,81.876925
Modification class,13477,81.387765
Modification,13473,81.363609
Post drying treatment,13264,80.101455


Duplicate rows: 0


In [35]:
#Rename columns
df = df.rename(columns={
    'Thermal conductivity W/(m K)': 'Thermal_conductivity',
    'Envelope density g/cm3': 'Density',
    'BET surface area m2/g': 'BET',
    'BJH pore size nm': 'Pore_size',
    'BJH pore volume (total) cm3/g': 'Pore_volume',
    'compression elastic modulus MPa': 'Elastic_modulus'
})

# Remove samples without a measured thermal conductivity (our target)
df = df.dropna(subset=['Thermal_conductivity'])

# Display categories and number of unique values
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(f"Number of categories: {df[col].nunique()}")
    print(df[col].unique()[:10])


--- Drying ---
Number of categories: 5
['SCD' 'FD' 'APD' 'VD' 'Foaming']

--- Purpose ---
Number of categories: 28
['Thermal insulation' 'Not specific'
 'Sound insulation, thermal insulation' 'Adsorbent' 'Filters' 'Separation'
 'Adsorbent (gas)' 'Thermoelectric devices' 'Steam generation'
 'Separation, thermal insulation']

--- Shape ---
Number of categories: 10
['Monolith' 'Fibers, monolith' 'Sheet' 'Fibers' 'Monolith, sheet'
 'Monolith (3D)' 'Beads, monolith' 'Beads' 'FIbers' 'Coating']

--- First materials ---
Number of categories: 90
['RF' 'Phenolic resin' 'Polyisocyanurate' 'Polyurethane'
 'Polydicyclopentadiene' 'Poly(urea-urethane)' 'Aramid fibers' 'Polyimide'
 'Lignocellulose' 'Cellulose']

--- Second materials ---
Number of categories: 123
[nan 'PVA, GO' 'Lignin' 'PMMA' 'Aramid' 'Silicone resin'
 'Silicone resin, cork' 'Phenolic resin' 'Starch' 'SiO2']

--- Modification ---
Number of categories: 63
[nan 'MTMS treatment' 'Fluorosilan modification' 'Carbonized'
 '3-Aminopropyl(

In [36]:

# Remove leading and trailing whitespace from categorical variables
categorical_columns = df.select_dtypes(include='object').columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

# Check for potential inconsistencies caused by capitalization
for col in categorical_columns:
    normalized = (
        df[col]
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

    print(
        f"{col}: "
        f"{df[col].nunique()} original -> "
        f"{normalized.nunique()} normalized categories"
    )



Drying: 5 original -> 5 normalized categories
Purpose: 28 original -> 27 normalized categories
Shape: 10 original -> 9 normalized categories
First materials: 90 original -> 90 normalized categories
Second materials: 123 original -> 122 normalized categories
Modification: 63 original -> 63 normalized categories
First material class: 16 original -> 16 normalized categories
Second materials class: 56 original -> 56 normalized categories
Modification class: 7 original -> 7 normalized categories
Gelation class: 9 original -> 9 normalized categories
Solvent class: 8 original -> 8 normalized categories
Process: 2193 original -> 2193 normalized categories
Gelation mechanism: 29 original -> 28 normalized categories
Final solvent: 31 original -> 31 normalized categories
Drying details: 12 original -> 12 normalized categories
Post drying treatment: 136 original -> 136 normalized categories
Post-drying cross-linking: 3 original -> 3 normalized categories
SEM: 20 original -> 20 normalized categorie

In [39]:
# Find categories that differ only by capitalization or whitespace

for col in categorical_columns:

    normalized = (
        df[col]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
    )

    temp = pd.DataFrame({
        'original': df[col],
        'normalized': normalized
    }).drop_duplicates()

    duplicates = (
        temp.groupby('normalized')['original']
        .agg(list)
    )

    duplicates = duplicates[
        duplicates.apply(lambda x: len(x) > 1)
    ]

    if not duplicates.empty:
        print(f"\n--- {col} ---")
        for normalized_value, original_values in duplicates.items():
            print(f"{original_values} -> '{normalized_value}'")


--- Purpose ---
['Separation, thermal insulation', 'Separation, thermal Insulation'] -> 'separation, thermal insulation'

--- Shape ---
['Fibers', 'FIbers'] -> 'fibers'

--- Second materials ---
['Aramid fibers, MXene', 'Aramid fibers, Mxene'] -> 'aramid fibers, mxene'

--- Gelation mechanism ---
['No gelation', 'NO gelation'] -> 'no gelation'

--- Thermal method detail ---
['Hot disk', 'hot disk'] -> 'hot disk'


In [40]:
# Correct inconsistent categorical values

corrections = {
    'Purpose': {
        'Separation, thermal Insulation': 'Separation, thermal insulation'
    },
    'Shape': {
        'FIbers': 'Fibers'
    },
    'Second materials': {
        'Aramid fibers, Mxene': 'Aramid fibers, MXene'
    },
    'Gelation mechanism': {
        'NO gelation': 'No gelation'
    },
    'Thermal method detail': {
        'hot disk': 'Hot disk'
    }
}

for col, replacements in corrections.items():
    df[col] = df[col].replace(replacements)

In [41]:
# Check categorical variables after corrections

for col in corrections:
    print(f"\n--- {col} ---")
    print(f"Number of categories: {df[col].nunique()}")
    print(df[col].value_counts())
    # Check for remaining capitalization inconsistencies

for col in categorical_columns:
    normalized = (
        df[col]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
    )

    print(
        f"{col}: "
        f"{df[col].nunique()} original categories -> "
        f"{normalized.nunique()} normalized categories"
    )


--- Purpose ---
Number of categories: 27
Purpose
Thermal insulation                               1808
Not specific                                      184
Steam generation                                   58
Separation, thermal insulation                     57
Sound insulation, thermal insulation               54
Separation                                         34
PCM support                                        33
Shield                                             33
Optoelectronic devices                             32
Sound insulation                                   25
Filters                                            23
Shield, thermal insulation                         19
Scaffold, thermal insulation                       18
Filters, thermal insulation                        16
Adsorbent, thermal insulation                      16
Thermoelectric devices                             15
Packaging                                          13
Adsorbent                       

In [37]:
# Check categorical variables
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(f"Number of categories: {df[col].nunique()}")
    print(df[col].value_counts().head(20))


--- Drying ---
Number of categories: 5
Drying
FD         1347
SCD         851
APD         276
VD            7
Foaming       3
Name: count, dtype: int64

--- Purpose ---
Number of categories: 28
Purpose
Thermal insulation                      1808
Not specific                             184
Steam generation                          58
Sound insulation, thermal insulation      54
Separation, thermal insulation            47
Separation                                34
PCM support                               33
Shield                                    33
Optoelectronic devices                    32
Sound insulation                          25
Filters                                   23
Shield, thermal insulation                19
Scaffold, thermal insulation              18
Filters, thermal insulation               16
Adsorbent, thermal insulation             16
Thermoelectric devices                    15
Packaging                                 13
Separation, thermal Insulation  

In [30]:
#Dataset after cleaning
print(f"Dataset after cleaning: {df.shape}")

missing = pd.DataFrame({
    'Missing values': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100
})
display(missing.sort_values('Missing (%)', ascending=False))


Dataset after cleaning: (2484, 28)


,Missing values,Missing (%)
Post-drying cross-linking,2454,98.792271
Pore_size,2324,93.558776
Modification class,2183,87.882448
Modification,2179,87.721417
Pore_volume,2166,87.198068
Post drying treatment,2002,80.595813
Elastic_modulus,1657,66.706924
Second materials,1596,64.251208
Second materials class,1596,64.251208
BET,1525,61.392915
